In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import normalize

BASE = Path.cwd().parent
DATA = BASE / "Datasets" / "processed" / "dataset.csv"

df = pd.read_csv(DATA).fillna({"title": "", "abstract": "", "ref_dois": ""})

MULTI_LABEL = True
if not MULTI_LABEL:
    df = df[df.n_labels == 1].reset_index(drop=True)

label_cols = [c for c in df.columns if c.startswith("lbl_")]
names = [c[4:] for c in label_cols]
Y = df[label_cols].values


dev_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.20, random_state=42, shuffle=True)
train_idx, val_idx = train_test_split(
    dev_idx, test_size=0.25, random_state=42, shuffle=True)   # 0.25 of 80% = 20%

print(f"train:      {len(train_idx):>5}  ({100*len(train_idx)/len(df):.0f}%)")
print(f"validation: {len(val_idx):>5}  ({100*len(val_idx)/len(df):.0f}%)")
print(f"test:       {len(test_idx):>5}  ({100*len(test_idx)/len(df):.0f}%)")
assert not (set(train_idx) & set(val_idx)) and not (set(train_idx) & set(test_idx))
print("\nno overlap between sets")

train:       4914  (60%)
validation:  1639  (20%)
test:        1639  (20%)

no overlap between sets


In [9]:


text = (df.title + " " + df.abstract).values

vectoriser = TfidfVectorizer(
    max_features=50000,     
    ngram_range=(1, 2),     
    min_df=2,               
    sublinear_tf=True,      
    stop_words="english",
)

X_train_text = vectoriser.fit_transform(text[train_idx])   
X_val_text   = vectoriser.transform(text[val_idx])         
X_test_text  = vectoriser.transform(text[test_idx])        

print(f"vocabulary learned from training data: {len(vectoriser.vocabulary_):,} terms")
print(f"train {X_train_text.shape} | val {X_val_text.shape} | test {X_test_text.shape}")

vocabulary learned from training data: 50,000 terms
train (4914, 50000) | val (1639, 50000) | test (1639, 50000)


In [10]:

vocabulary, row_idx, col_idx = {}, [], []
for i, refs in enumerate(df.ref_dois):
    for r in (x for x in refs.split("|") if x):
        j = vocabulary.setdefault(r, len(vocabulary))
        row_idx.append(i)
        col_idx.append(j)

X_cite = normalize(csr_matrix((np.ones(len(row_idx)), (row_idx, col_idx)),
                              shape=(len(df), len(vocabulary))))
print(f"citation matrix: {X_cite.shape[0]} papers x {X_cite.shape[1]} references")

citation matrix: 8192 papers x 367569 references


In [11]:
def evaluate(y_true, pred):
    return (f1_score(y_true, pred, average="micro", zero_division=0),
            f1_score(y_true, pred, average="macro", zero_division=0))

print("choosing C on the validation set:")
best_C, best = None, -1
for C in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    m = OneVsRestClassifier(LinearSVC(C=C, max_iter=3000), n_jobs=-1)
    m.fit(X_train_text, Y[train_idx])
    micro = evaluate(Y[val_idx], m.predict(X_val_text))[0]
    print(f"   C={C:<4} val micro-F1 {micro:.4f}")
    if micro > best:
        best, best_C = micro, C

print(f"\ngrid best: C={best_C}")


best_C = 2.0
print(f"selected for all experiments: C={best_C}")

text_model = OneVsRestClassifier(LinearSVC(C=best_C, max_iter=3000), n_jobs=-1)
text_model.fit(X_train_text, Y[train_idx])
pred_text = text_model.predict(X_test_text)

print(f"\nMODEL 1 — TEXT, on test:")
print("  micro-F1 {:.3f}   macro-F1 {:.3f}".format(*evaluate(Y[test_idx], pred_text)))

choosing C on the validation set:
   C=0.1  val micro-F1 0.6280
   C=0.5  val micro-F1 0.7246
   C=1.0  val micro-F1 0.7306
   C=2.0  val micro-F1 0.7362
   C=5.0  val micro-F1 0.7383
   C=10.0 val micro-F1 0.7386

grid best: C=10.0
selected for all experiments: C=2.0

MODEL 1 — TEXT, on test:
  micro-F1 0.733   macro-F1 0.727


In [12]:
def citation_scores(fit_idx, score_idx, k):
    
    S = (X_cite[score_idx] @ X_cite[fit_idx].T).toarray()
    kk = min(k, S.shape[1] - 1)
    nn = np.argpartition(-S, kk, axis=1)[:, :kk]
    out = np.zeros((len(score_idx), Y.shape[1]))
    for i in range(len(score_idx)):
        w = S[i, nn[i]]
        if w.sum() > 0:
            out[i] = (Y[fit_idx][nn[i]] * w[:, None]).sum(0) / w.sum()
    return out


def to_labels(scores, threshold):
    
    pred = (scores >= threshold).astype(int)
    empty = pred.sum(1) == 0
    pred[empty, scores[empty].argmax(1)] = 1
    return pred


print("choosing k and the threshold on the validation set:")
best_k, best_thr, best = None, None, -1
for k in [5, 10, 20, 30, 50]:
    s_val = citation_scores(train_idx, val_idx, k)
    for thr in [0.20, 0.25, 0.30, 0.35, 0.40]:
        micro = evaluate(Y[val_idx], to_labels(s_val, thr))[0]
        if micro > best:
            best, best_k, best_thr = micro, k, thr
    print(f"   k={k:<4} best val micro-F1 so far {best:.4f}")

print(f"\nchosen k={best_k}, threshold={best_thr}")

scores_cite_test = citation_scores(train_idx, test_idx, best_k)
pred_cite = to_labels(scores_cite_test, best_thr)

print(f"\nMODEL 2 — CITATION, on test:")
print("  micro-F1 {:.3f}   macro-F1 {:.3f}".format(*evaluate(Y[test_idx], pred_cite)))

choosing k and the threshold on the validation set:
   k=5    best val micro-F1 so far 0.7520
   k=10   best val micro-F1 so far 0.7580
   k=20   best val micro-F1 so far 0.7605
   k=30   best val micro-F1 so far 0.7605
   k=50   best val micro-F1 so far 0.7605

chosen k=20, threshold=0.35

MODEL 2 — CITATION, on test:
  micro-F1 0.751   macro-F1 0.751


In [13]:
def zscale(a):
   
    return (a - a.mean()) / (a.std() + 1e-9)



scores_text_val = text_model.decision_function(X_val_text)
scores_cite_val = citation_scores(train_idx, val_idx, best_k)

print("choosing hybrid weight and threshold on the validation set:")
best_w, best_hthr, best = 0.5, 0.0, -1
for w in [0.0, 0.2, 0.35, 0.5, 0.65, 0.8, 1.0]:
    combined = (1 - w) * zscale(scores_text_val) + w * zscale(scores_cite_val)
    for thr in [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]:
        micro = evaluate(Y[val_idx], to_labels(combined, thr))[0]
        if micro > best:
            best, best_w, best_hthr = micro, w, thr

print(f"chosen w={best_w} (weight on citations), threshold={best_hthr}")

scores_text_test = text_model.decision_function(X_test_text)
combined_test = (1 - best_w) * zscale(scores_text_test) + best_w * zscale(scores_cite_test)
pred_hybrid = to_labels(combined_test, best_hthr)

print(f"\nMODEL 3 — HYBRID, on test:")
print("  micro-F1 {:.3f}   macro-F1 {:.3f}".format(*evaluate(Y[test_idx], pred_hybrid)))

choosing hybrid weight and threshold on the validation set:
chosen w=0.35 (weight on citations), threshold=1.0

MODEL 3 — HYBRID, on test:
  micro-F1 0.775   macro-F1 0.774


In [14]:
print("All three models, evaluated on the held-out test set")
print("(hyperparameters chosen on validation, test used once)\n")
print(f"{'model':<12}{'micro-F1':>10}{'macro-F1':>10}")
for label, pred in [("text", pred_text), ("citation", pred_cite), ("hybrid", pred_hybrid)]:
    print(f"{label:<12}{f1_score(Y[test_idx], pred, average='micro'):>10.3f}"
          f"{f1_score(Y[test_idx], pred, average='macro'):>10.3f}")

print("\nper-field F1:")
print(f"{'field':<20}{'text':>8}{'citation':>10}{'hybrid':>8}")
f_t = f1_score(Y[test_idx], pred_text, average=None, zero_division=0)
f_c = f1_score(Y[test_idx], pred_cite, average=None, zero_division=0)
f_h = f1_score(Y[test_idx], pred_hybrid, average=None, zero_division=0)
for j, n in enumerate(names):
    print(f"{n:<20}{f_t[j]:>8.3f}{f_c[j]:>10.3f}{f_h[j]:>8.3f}")

All three models, evaluated on the held-out test set
(hyperparameters chosen on validation, test used once)

model         micro-F1  macro-F1
text             0.733     0.727
citation         0.751     0.751
hybrid           0.775     0.774

per-field F1:
field                   text  citation  hybrid
Biotechnology          0.825     0.807   0.879
Chemistry              0.812     0.820   0.822
CS_AI                  0.645     0.711   0.730
CS_InfoSystems         0.571     0.655   0.635
Engineering_EE         0.659     0.681   0.725
Environmental          0.732     0.728   0.789
MaterialsScience       0.755     0.783   0.803
Mathematics            0.768     0.760   0.814
Physics_Applied        0.605     0.686   0.657
Psychology             0.895     0.877   0.886
